<a href="https://colab.research.google.com/github/zeynpakn/LunarLander-PPO-Reinforcement-Learning/blob/main/LunarLander_PPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ortam hazirligi, kurulum ve video uretimi icin gerekli sist. aracları
!apt-get update -qq
!apt-get install -y -qq swig cmake ffmpeg

# LunarLander-v2, PPO ve Hugging Face kutuphaneleri
!pip install -q "gymnasium[box2d]==0.29.1" "stable-baselines3[extra]==2.6.0" huggingface_sb3


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package swig.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../swig_4.2.0-2ubuntu1_amd64.deb ...
Unpacking swig (4.2.0-2ubuntu1) ...
Setting up swig (4.2.0-2ubuntu1) ...
Processing triggers for man-db (2.12.0-4build2) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.4/374.4 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.5/184.5 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into accou

In [2]:
# gerekli kutuphaneleri ice aktariyoruz

import os
import gymnasium as gym

from IPython.display import Video, display                        # video oynatma

from stable_baselines3 import PPO                                 # PPO egitim algoritmasi
from stable_baselines3.common.env_util import make_vec_env        # paralel ortam olusturma
from stable_baselines3.common.evaluation import evaluate_policy   # model basarisini olcme
from stable_baselines3.common.monitor import Monitor              # oyun sonuclarini izleme

from huggingface_hub import notebook_login, whoami                # hesaba giris islemleri
from huggingface_sb3 import package_to_hub                        # modelin hub'a yuklenmesi

print("Kutuphaneler basariyla yuklendi!")
print("Gymnasium:", gym.__version__)


Kutuphaneler basariyla yuklendi!
Gymnasium: 0.29.1


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
# LunarLander-v2 ortamini olusturuyoruz
env = gym.make("LunarLander-v2")

# ortami baslangic durumuna getiriyoruz
observation, info = env.reset()

# ortam bilgilerini inceliyoruz
print("Ilk gozlem:", observation)
print("Gozlem boyutu:", env.observation_space.shape)
print("Eylem uzayi:", env.action_space)
print("Eylem sayisi:", env.action_space.n)

env.close()


Ilk gozlem: [-0.00549622  1.4222345  -0.5567185   0.50284827  0.00637548  0.12610501
  0.          0.        ]
Gozlem boyutu: (8,)
Eylem uzayi: Discrete(4)
Eylem sayisi: 4


/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.13/dist-packages/pkg_resources/__init__.py:3148: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('sphinxcontrib')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC:

In [ ]:
# Observation space'den aldığımız 8 değer şunları ifade eder: [0,1,2,3,4,5,6,7]
# 0.   X konumu (yatay)
# 1.   Y konumu (dikey)
# 2.   X hız (yatay)
# 3.   Y hız (dikey)
# 4.   Aracın açısı
# 5.   Aracın hızı (dönme)
# 6.   Sol ayak teması (0/1)
# 7.   Sağ ayak teması (0/1)

# Action Space'de ise ajanın seçebileceği 4 eylem var:
# 0.   Motor kapalı
# 1.   Sol motor
# 2.   Ana motor
# 3.   Sağ motor

In [4]:
# ortami olusturuyoruz
env = gym.make("LunarLander-v2")
observation, info = env.reset()

total_reward = 0

for step in range(1000):

    # rastgele bir eylem seciyoruz
    action = env.action_space.sample()

    # eylemi gerceklestiriyoruz
    observation, reward, terminated, truncated, info = env.step(action)

    # odulleri topluyoruz
    total_reward += reward

    # ilk 10 adimi yazdiriyoruz
    if step < 10:
        print(f"Adım: {step+1}, Eylem: {action}, Ödül: {reward:.2f}")

    # oyun bittiyse donguden cikiyoruz
    if terminated or truncated:
        break

print("\nToplam adim:", step + 1)
print("Toplam odul:", round(total_reward, 2))

env.close()


Adım: 1, Eylem: 3, Ödül: -2.06
Adım: 2, Eylem: 3, Ödül: -2.62
Adım: 3, Eylem: 1, Ödül: -0.36
Adım: 4, Eylem: 2, Ödül: -0.24
Adım: 5, Eylem: 2, Ödül: -0.58
Adım: 6, Eylem: 2, Ödül: -1.09
Adım: 7, Eylem: 0, Ödül: -1.24
Adım: 8, Eylem: 3, Ödül: -2.39
Adım: 9, Eylem: 1, Ödül: -0.27
Adım: 10, Eylem: 1, Ödül: -0.26

Toplam adim: 78
Toplam odul: -280.5


In [ ]:
# ajan rastgele eylemler secerek 78 adimda -280.5 odul aldi
# ogrenme gerceklestirmedik burada sadece secim yaptirdik

In [5]:
# egitim icin 16 ortam olusturuyoruz
env = make_vec_env("LunarLander-v2", n_envs=16)

# PPO modelini tanimliyoruz
model = PPO(
    policy="MlpPolicy",    # sinir agi tabanli politika
    env=env,               # egitim ortami
    n_steps=1024,          # her ortamdan toplanacak adim
    batch_size=64,         # egitim grup boyutu
    n_epochs=4,            # egitim tekrar sayisi
    gamma=0.999,           # gelecek odullerin agirligi
    gae_lambda=0.98,       # avantaj tahmini katsayisi
    ent_coef=0.01,         # kesif yapmayi tesvik eder
    verbose=1,             # egitim bilgilerini goster
    device="cpu"           # CPU ile egitim
)

print("PPO modeli olusturuldu.")


Using cpu device
PPO modeli olusturuldu.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
# PPO modelini 1 milyon adim egitiyoruz
model.learn(total_timesteps=1_000_000)

# egitilen modeli kaydediyoruz
model_name = "ppo-LunarLander-v2"
model.save(model_name)

print("Egitim tamamlandi.")


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 90.9     |
|    ep_rew_mean     | -191     |
| time/              |          |
|    fps             | 3916     |
|    iterations      | 1        |
|    time_elapsed    | 4        |
|    total_timesteps | 16384    |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 89.4         |
|    ep_rew_mean          | -153         |
| time/                   |              |
|    fps                  | 2418         |
|    iterations           | 2            |
|    time_elapsed         | 13           |
|    total_timesteps      | 32768        |
| train/                  |              |
|    approx_kl            | 0.0066025164 |
|    clip_fraction        | 0.0343       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.38        |
|    explained_variance   | 0.00184      |
|    learning_r

In [7]:
# test icin ayri ortam olusturuyoruz
eval_env = Monitor(gym.make("LunarLander-v2"))

# modeli 10 oyunda test ediyoruz
mean_reward, std_reward = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=10,
    deterministic=True
)

print("Ortalama odul:", mean_reward)
print("Standart sapma:", std_reward)
print("Sertifika skoru:", mean_reward - std_reward)

eval_env.close()


Ortalama odul: 250.74029430000002
Standart sapma: 23.296385854032085
Sertifika skoru: 227.44390844596793


In [10]:
# video klasorunu olusturuyoruz
os.makedirs("videos", exist_ok=True)

# goruntu alabilen test ortami olusturuyoruz
video_env = gym.make("LunarLander-v2", render_mode="rgb_array")
video_env = gym.wrappers.RecordVideo(
    video_env,
    video_folder="videos",
    episode_trigger=lambda episode_id: True
)

# oyunu baslatiyoruz
obs, info = video_env.reset()
terminated, truncated = False, False

while not (terminated or truncated):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = video_env.step(int(action))

video_env.close()

# kaydedilen videoyu izliyoruz
video_path = "videos/" + sorted(
    f for f in os.listdir("videos") if f.endswith(".mp4")
)[-1]

display(Video(video_path, embed=True))


Moviepy - Building video /content/videos/rl-video-episode-0.mp4.
Moviepy - Writing video /content/videos/rl-video-episode-0.mp4



Moviepy - Done !
Moviepy - video ready /content/videos/rl-video-episode-0.mp4


In [ ]:
# rastgele ajan 78 adimda toplam -280.5 odul aldi.
# PPO ile egitilen ajan 10 oyunda ortalama 250.74 odul aldi.
# standart sapma 23.30, sertifika skoru 227.44 oldu.
# egitim sonrasi ajan daha basarili inis yapmayi ogrendi.

In [ ]:
# modelimizi egittik, nasil oldugunu gorduk
# huggingface'e modeli yuklicez
# giris yapabilmek icin "https://huggingface.co/settings/tokens"
# sitesinden write olarak token olusturmali ve token'i girmeniz gerekir

In [11]:
# Hugging Face hesabina giris yapiyoruz
notebook_login()


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [12]:
# test ortamini uyumlu hale getirdik
from stable_baselines3.common.vec_env import DummyVecEnv

# Model bilgilerini tanimliyoruz
username = whoami()["name"]
model_name = "ppo-LunarLander-v2"
env_id = "LunarLander-v2"
model_architecture = "PPO"

repo_id = f"{username}/{model_name}"
commit_message = "Upload PPO LunarLander-v2 trained agent"

# yukleme icin test ortami olusturuyoruz
eval_env = DummyVecEnv([
    lambda: Monitor(gym.make(env_id, render_mode="rgb_array"))
])

# modeli Hugging Face'e yukluyoruz
package_to_hub(
    model=model,
    model_name=model_name,
    model_architecture=model_architecture,
    env_id=env_id,
    eval_env=eval_env,
    repo_id=repo_id,
    commit_message=commit_message
)

print("Model basariyla yuklendi!!")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


ℹ This function will save, evaluate, generate a video of your agent,
create a model card and push everything to the hub. It might take up to 1min.
This is a work in progress: if you encounter a bug, please open an issue.
Saving video to /tmp/tmp4p_cn920/-step-0-to-step-1000.mp4
Moviepy - Building video /tmp/tmp4p_cn920/-step-0-to-step-1000.mp4.
Moviepy - Writing video /tmp/tmp4p_cn920/-step-0-to-step-1000.mp4



Moviepy - Done !
Moviepy - video ready /tmp/tmp4p_cn920/-step-0-to-step-1000.mp4
✘ 'DummyVecEnv' object has no attribute 'video_recorder'
✘ We are unable to generate a replay of your agent, the package_to_hub
process continues
✘ Please open an issue at
https://github.com/huggingface/huggingface_sb3/issues
ℹ Pushing repo zeynpakn/ppo-LunarLander-v2 to the Hugging Face Hub


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

  ...y0/ppo-LunarLander-v2.zip: 100%|##########|  153kB /  153kB            

  ...r-v2/policy.optimizer.pth: 100%|##########| 88.4kB / 88.4kB            

  ...LunarLander-v2/policy.pth: 100%|##########| 44.0kB / 44.0kB            

  ...-v2/pytorch_variables.pth: 100%|##########| 1.26kB / 1.26kB            

ℹ Your model is pushed to the Hub. You can view your model here:
https://huggingface.co/zeynpakn/ppo-LunarLander-v2/tree/main/
Model basariyla yuklendi!!
